# Train the gas-cylinder detector with YOLO26 on Google Colab

This notebook fine-tunes YOLO26s (small) on our gas-cylinder dataset using a free Colab GPU,
which is much faster than training on a laptop CPU. It has the same steps and settings as
`train_colab.ipynb`, which trained the first YOLOv8n model.

Before you start:
1. Put the dataset zip in your Google Drive and set `DATASET_ZIP` in cell 4 to its path.
2. Check that **Runtime → Change runtime type** is set to **T4 GPU**.

Then run every cell from top to bottom with **Runtime → Run all** and keep this tab open.

## 1. Check the GPU

`nvidia-smi` should show a **Tesla T4**. If it says `command not found`, this runtime has
no GPU: choose **Runtime → Change runtime type → T4 GPU** and run the notebook again.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

The same version as `requirements.txt` in the repository; it already includes YOLO26.

In [ ]:
!pip install -q "ultralytics~=8.4.171"

## 3. Connect Google Drive

Colab asks for permission to access your Drive. The dataset is read from Drive, and the
training results are written back to Drive, so they are kept even if Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Unzip the dataset

The zip is copied to the Colab disk first, because training straight from Drive is very slow.
The zip must contain `data.yaml` and the `train`, `valid` and `test` folders, at its root or in one folder.
At the end you should see the path of `data.yaml` and the folders next to it.

In [ ]:
from pathlib import Path

# TODO: set this to the dataset zip in your Google Drive.
DATASET_ZIP = Path("/content/drive/MyDrive/PUT_DATASET_ZIP_HERE.zip")
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"Dataset zip not found, set DATASET_ZIP: {DATASET_ZIP}")

!cp "{DATASET_ZIP}" /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/data
DATA_YAML = next(Path("/content/data").rglob("data.yaml"))  # Works with or without a top folder in the zip.
print(DATA_YAML)
!ls "{DATA_YAML.parent}"

## 5. Train

Same settings as the first Colab model, but starting from the pre-trained YOLO26 small model.

- `MODEL` is the pre-trained model. Use `"yolo26n.pt"` for the nano model, which is faster but usually less accurate.
- `time=0.75` stops training after 45 minutes. Ultralytics measures the first epoch and then
  trains as many epochs as fit. The small model needs more time per epoch than nano, so increase it
  if you have more time and want a better model.
- `patience=10` stops earlier when the validation score has not improved for 10 epochs.
- After every epoch, the weights are saved to `My Drive/gas_cylinders_runs/yolo26s/weights/`.

In [ ]:
from ultralytics import YOLO

MODEL = "yolo26s.pt"  # Or "yolo26n.pt" for the faster nano model.

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    project="/content/drive/MyDrive/gas_cylinders_runs",  # Save to Drive, not to the Colab disk.
    name=Path(MODEL).stem,  # yolo26s, so these runs are kept apart from the YOLOv8n runs.
    time=0.75,  # Train for at most 45 minutes; this overrides epochs.
    patience=10,  # Stop after 10 epochs without validation improvement.
    imgsz=480,  # Same image size as the first Colab model.
    batch=32,  # The GPU has enough memory for bigger batches than a laptop.
    cache=True,  # Keep the images in RAM, because Colab has few CPUs to read the JPGs.
)
print(f"Training results: {model.trainer.save_dir}")
print(f"Best weights: {model.trainer.best}")

## 6. Look at the results

`results.png` shows the losses and the validation scores after every epoch.
The losses should go down, while precision, recall and mAP should go up.

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{model.trainer.save_dir}/results.png"))

## 7. Use the model on your laptop

1. In Google Drive, open `gas_cylinders_runs/yolo26s/weights/` and download `best.pt`.
   If you trained more than once, the folder is `yolo26s-2`, `yolo26s-3`, ... (cell 5 prints the exact path).
2. Put it in the repository under `src/runs/`, which is ignored by Git, for example `src/runs/yolo26s/best.pt`.
   If the team selects it as a shared model, name it with the date, trainer and descriptor, for example
   `2026-11-02_cosmin_cylinders-yolo26s-60e.pt`, and add it to `src/models/` as the README's Models section explains.
3. From the repository root, evaluate it on the test split and track the cylinders in a video:

```bash
python src/scripts/evaluate.py --weights src/runs/yolo26s/best.pt --data <path to data.yaml>
python src/scripts/track.py --weights src/runs/yolo26s/best.pt --source <path to video>
```

**If Colab disconnects during training**, run cells 1-4 again, then continue from the last epoch:

```python
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/gas_cylinders_runs/yolo26s/weights/last.pt")
model.train(resume=True)
```